##A Multi-Objective Optimization Framework for Resilient Infrastructure Asset Management<br>
Code, data, and supplemental materials<br>
Fredy Díaz-Durán · ORCID [0000-0001-5344-5466](https://orcid.org/0000-0001-5344-5466) · [diazdura@ualberta.ca](mailto:diazdura@ualberta.ca) · [fadiazdu@uwaterloo.ca](mailto:fadiazdu@uwaterloo.ca)<br>
Department of Civil and Environmental Engineering, University of Alberta, Edmonton, AB, Canada<br>
Department of Civil and Environmental Engineering, University of Waterloo, Waterloo, ON, Canada<br>
DOI: [10.5281/zenodo.22973335](https://doi.org/10.5281/zenodo.22973335)<br>
Licenses. Code: MIT (`LICENSE`). Documents, figures, and outputs: CC BY 4.0 (`LICENSE-CC-BY-4.0.md`). NBI files: public domain (`DATA_NOTICE.md`).<br>
This notebook runs in Google Colab or in a local Jupyter session. Every run downloads the public FHWA National Bridge Inventory files and reproduces every number, table, and figure of the paper and its supplement from them.

## 1. Locate the project folder
In Colab, set `PROJECT_DIR` to the Drive folder that contains `run_all.py`. Locally, open this notebook from that folder; no change is needed.

In [1]:
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/Colab Notebooks/[Research_Papers]/06_Optimization_Framework_Asset_Management/ResilientIAM_Framework'   # <-- EDIT: folder containing run_all.py
    os.chdir(PROJECT_DIR)
assert os.path.exists('run_all.py'), f'run_all.py not found in {os.getcwd()}: set PROJECT_DIR (Colab) or open the notebook from the project folder'
print('Project folder:', os.getcwd(), '| environment:', 'Google Colab' if IN_COLAB else 'local')

Mounted at /content/drive
Project folder: /content/drive/MyDrive/Colab Notebooks/[Research_Papers]/06_Optimization_Framework_Asset_Management/ResilientIAM_Framework | environment: Google Colab


## 2. Install dependencies

In [2]:
%pip install -q -r requirements.txt


## 3. Run the complete analysis
Downloads the NBI files from FHWA, verifies them, builds every model input from them, and runs every analysis, figure, and table (about 100 minutes). Only if the FHWA site is unavailable, add `--offline` to use archived copies placed in `data/raw` (hash-verified). **If the run is interrupted, rerun cells 1-3:** the inputs are rebuilt from a fresh download and the run continues after the last completed analysis.

In [3]:
!{sys.executable} -u run_all.py

A Multi-Objective Optimization Framework for Resilient Infrastructure Asset Management
Code, data, and supplemental materials
Fredy Díaz-Durán · ORCID 0000-0001-5344-5466 · diazdura@ualberta.ca · fadiazdu@uwaterloo.ca
Department of Civil and Environmental Engineering, University of Alberta, Edmonton, AB, Canada
Department of Civil and Environmental Engineering, University of Waterloo, Waterloo, ON, Canada
DOI: 10.5281/zenodo.22973335
Licenses. Code: MIT (LICENSE). Documents, figures, and outputs: CC BY 4.0 (LICENSE-CC-BY-4.0.md). NBI files: public domain (DATA_NOTICE.md).

Environment recorded in outputs/environment.json: python 3.13.15, numpy 2.1.3, pandas 2.2.3, scipy 1.16.3, pulp 3.3.2
DATA 1  PUBLIC FHWA FILES: DOWNLOAD AND VERIFY (before any analysis)
  2019: 337,655 bytes from https://www.fhwa.dot.gov/bridge/nbi/2019/delimited/RI19.txt -> matches reference
  2020: 336,585 bytes from https://www.fhwa.dot.gov/bridge/nbi/2020/delimited/RI20.txt -> matches reference
  2021: 337,363 b

## 4. Build the Zenodo archive
Packages the code, the six NBI files this run downloaded (with their SHA-256 manifest), the outputs, the paper and supplement figures, and `results_summary.txt` into `ResilientIAM_Framework_v1.0.0_Zenodo.zip`. Run it after a complete run. Checkpoints, caches and intermediate model inputs are excluded (every run rebuilds them). The Supplemental Materials PDF is uploaded to Zenodo as a separate file.

In [4]:
import glob, hashlib, zipfile, json
import paper_figures as F
NAME = 'ResilientIAM_Framework_v1.0.0'
env = json.load(open('outputs/environment.json'))
man = json.load(open('data/raw/download_manifest.json'))
raw = sorted(glob.glob('data/raw/RI20*.csv'))
assert len(raw) == 6, f'expected 6 NBI files in data/raw, found {len(raw)}: run the complete analysis first'
recs = man['files'] if isinstance(man, dict) and 'files' in man else man
ref = {str(r['year']): r['sha256'] for r in (recs.values() if isinstance(recs, dict) else recs)}
for f in raw:                                    # the archived NBI files must be the verified ones
    year = os.path.basename(f)[2:6]
    assert hashlib.sha256(open(f, 'rb').read()).hexdigest() == ref[year], f'{f} does not match the reference hash'
files = sorted(glob.glob('*.py')) + ['run_all.ipynb', 'README.md', 'LICENSE', 'LICENSE-CC-BY-4.0.md', 'CITATION.cff',
         'DATA_NOTICE.md', 'requirements.txt', 'results_summary.txt', 'data/raw/download_manifest.json'] + raw
files += sorted(glob.glob('outputs/*'))
files += [f'figures/{n}.{e}' for n in F.MAIN + F.SUPPLEMENT for e in ('pdf', 'png') if os.path.exists(f'figures/{n}.{e}')]
missing = [f for f in files if not os.path.exists(f)]
assert not missing, f'missing: {missing}'
with zipfile.ZipFile(f'{NAME}_Zenodo.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, f'{NAME}/{f}')
print(f'{NAME}_Zenodo.zip: {len(files)} files, {os.path.getsize(NAME + "_Zenodo.zip") / 1e6:.1f} MB')
print('code fingerprint of this run:', env['code_signature'], '| NBI files verified against the reference hashes: 6 of 6')
if IN_COLAB:
    from google.colab import files as colab_files
    colab_files.download(f'{NAME}_Zenodo.zip')


ResilientIAM_Framework_v1.0.0_Zenodo.zip: 107 files, 18.9 MB
code fingerprint of this run: dedd54086c043434 | NBI files verified against the reference hashes: 6 of 6


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>